In [ ]:
%ip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.5 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transform

In [2]:
import os
import json
import glob
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor
import math

In [3]:
# !apt install tree
# !tree -d /kaggle/input/datasets/awsaf49/coco-2017-dataset

In [4]:
import kagglehub
MODEL_ID = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b")

In [5]:
import gc
import torch

# Delete model and tokenizer if they still exist in the namespace
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

# Force Python garbage collection
gc.collect()

# Clear PyTorch's internal VRAM cache
torch.cuda.empty_cache()

print("Memory cleared!")


Memory cleared!


In [6]:
max_memory_mapping = {
    0: "15GiB",  # Maximize VRAM usage on GPU 0
    1: "15GiB",  # Maximize VRAM usage on GPU 1
    "cpu": "4GiB" # Severely restrict CPU RAM to force GPU offloading
}

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory=max_memory_mapping
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print("Model successfully loaded across devices:")
print(model.hf_device_map)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Model successfully loaded across devices:
{'model.vision_tower': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 1, 'model.language_model.layers.5': 1, 'model.language_model.layers.6': 1, 'model.language_model.layers.7': 1, 'model.language_model.layers.8': 1, 'model.language_model.layers.9': 1, 'model.language_model.layers.10': 1, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.layers.23': 1, 

In [7]:
json_matches = glob.glob("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json", recursive=True)
if not json_matches:
    raise FileNotFoundError("Could not find captions_train2017.json in /kaggle/input/")
CAPTIONS_JSON_PATH = json_matches[0]

MODEL_DIR = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b/1"
OUTPUT_DIR = Path("/kaggle/working/coco_gemma_layer40_train")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading captions from: {CAPTIONS_JSON_PATH}")
print(f"Saving shards to: {OUTPUT_DIR}")

Reading captions from: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json
Saving shards to: /kaggle/working/coco_gemma_layer40_train


In [8]:
@torch.inference_mode()
def extract_penultimate_embeddings(
    captions: list[str], 
    tokenizer, 
    model, 
    max_length: int = 64
) -> list[torch.Tensor]:

    primary_device = model.device

    inputs = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(primary_device)

    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        output_hidden_states=True
    )

    penultimate = outputs.hidden_states[-2].detach().cpu()
    mask = inputs["attention_mask"].detach().cpu().bool()

    unpadded_embeddings = []
    for hidden_state, attn_mask in zip(penultimate, mask):
        valid_tokens = hidden_state[attn_mask]
        unpadded_embeddings.append(valid_tokens)

    return unpadded_embeddings

In [9]:
with open(CAPTIONS_JSON_PATH, "r") as f:
    coco_data = json.load(f)

# Create a mapping of image_id to a LIST of all its captions
image_to_captions = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in image_to_captions:
        image_to_captions[img_id] = []
    image_to_captions[img_id].append(ann["caption"])

img_ids = list(image_to_captions.keys())
total_samples = len(img_ids)
img_caps = list(image_to_captions.values())
total_caps_samples = len(img_caps) * len(image_to_captions[img_ids[0]])
print(f"Total unique images to process: {total_samples}")
print(f"Total unique captions to process: {total_caps_samples}")

Total unique images to process: 118287
Total unique captions to process: 591435


In [10]:
CHUNK_ID = 3
NUM_CHUNKS = 4

# Calculate indices to slice the dataset
chunk_size = math.ceil(total_samples / NUM_CHUNKS)
start_idx = CHUNK_ID * chunk_size
end_idx = min(start_idx + chunk_size, total_samples)

# Slice the image IDs so this specific notebook only processes its assigned portion
chunk_img_ids = img_ids[start_idx:end_idx]
chunk_total = len(chunk_img_ids)
print(f"Notebook {CHUNK_ID} processing indices {start_idx} to {end_idx} ({chunk_total} images)")

# Update output directory to keep chunks separate
OUTPUT_DIR = Path(f"/kaggle/working/coco_gemma_layer40_train_chunk_{CHUNK_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 2
SHARD_SIZE = 10000  

current_shard = {}
shard_idx = 0

for i in tqdm(range(0, chunk_total, BATCH_SIZE), desc=f"Chunk {CHUNK_ID} (5x Captions)"):
    batch_ids = chunk_img_ids[i : i + BATCH_SIZE]
    
    flat_caps = []
    caps_per_image = []
    for img_id in batch_ids:
        caps = image_to_captions[img_id]
        flat_caps.extend(caps)
        caps_per_image.append(len(caps))
        
    # Extract all text embeddings in one batch
    flat_tensors = extract_penultimate_embeddings(flat_caps, tokenizer, model)
    
    tensor_idx = 0
    for img_id, num_caps in zip(batch_ids, caps_per_image):
        current_shard[img_id] = flat_tensors[tensor_idx : tensor_idx + num_caps]
        tensor_idx += num_caps

    if len(current_shard) >= SHARD_SIZE or (i + BATCH_SIZE) >= chunk_total:
        # Appended the CHUNK_ID to the filename for safe merging later
        shard_file = OUTPUT_DIR / f"gemma_chunk_{CHUNK_ID}_shard_{shard_idx:03d}.pt"
        torch.save(current_shard, shard_file)
        shard_idx += 1
        current_shard = {}

print(f"\nDone! Successfully saved {shard_idx} shards to {OUTPUT_DIR}")

Notebook 3 processing indices 88716 to 118287 (29571 images)


Chunk 3 (5x Captions):   0%|          | 0/14786 [00:00<?, ?it/s]


Done! Successfully saved 3 shards to /kaggle/working/coco_gemma_layer40_train_chunk_3


In [11]:
json_matches = glob.glob("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_val2017.json", recursive=True)
if not json_matches:
    raise FileNotFoundError("Could not find captions_val2017.json in /kaggle/input/")
CAPTIONS_JSON_PATH = json_matches[0]

OUTPUT_DIR = Path("/kaggle/working/coco_gemma_layer40_val")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading captions from: {CAPTIONS_JSON_PATH}")
print(f"Saving shards to: {OUTPUT_DIR}")

Reading captions from: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_val2017.json
Saving shards to: /kaggle/working/coco_gemma_layer40_val


In [12]:
with open(CAPTIONS_JSON_PATH, "r") as f:
    coco_data = json.load(f)

# Create a mapping of image_id to a LIST of all its captions
image_to_captions = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in image_to_captions:
        image_to_captions[img_id] = []
    image_to_captions[img_id].append(ann["caption"])

img_ids = list(image_to_captions.keys())
total_samples = len(img_ids)
img_caps = list(image_to_captions.values())
total_caps_samples = len(img_caps) * len(image_to_captions[img_ids[0]])
print(f"Total unique images to process: {total_samples}")
print(f"Total unique captions to process: {total_caps_samples}")

Total unique images to process: 5000
Total unique captions to process: 25000


In [13]:
# BATCH_SIZE of 2 images will process roughly 10 captions per forward pass
BATCH_SIZE = 2
SHARD_SIZE = 10000  

current_shard = {}
shard_idx = 0

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Extracting 5x Gemma Captions"):
    batch_ids = img_ids[i : i + BATCH_SIZE]
    
    # Flatten the captions into a single list for the tokenizer
    flat_caps = []
    caps_per_image = []
    for img_id in batch_ids:
        caps = image_to_captions[img_id]
        flat_caps.extend(caps)
        caps_per_image.append(len(caps))
        
    # Extract all text embeddings in one batch
    flat_tensors = extract_penultimate_embeddings(flat_caps, tokenizer, model)
    
    # Group the extracted tensors back into a list for their respective image IDs
    tensor_idx = 0
    for img_id, num_caps in zip(batch_ids, caps_per_image):
        current_shard[img_id] = flat_tensors[tensor_idx : tensor_idx + num_caps]
        tensor_idx += num_caps

    if len(current_shard) >= SHARD_SIZE or (i + BATCH_SIZE) >= total_samples:
        shard_file = OUTPUT_DIR / f"gemma_embeddings_shard_{shard_idx:03d}.pt"
        torch.save(current_shard, shard_file)
        shard_idx += 1
        current_shard = {}

print(f"\nDone! Successfully saved {shard_idx} shards to {OUTPUT_DIR}")

Extracting 5x Gemma Captions:   0%|          | 0/2500 [00:00<?, ?it/s]


Done! Successfully saved 1 shards to /kaggle/working/coco_gemma_layer40_val
